# Przypisanie PS do budynków

Celem jest przygotowanie warstwy budynków, którą będzie można wykorzystać do dalszych analiz przestrzennych i wskazywania obiektów wymagających dokładniejszej obserwacji.

## Wczytanie danych

- EGMS_Bytom_results.csv – punkty EGMS z cechami przemieszczeń oraz wynikami Isolation Forest i K-means;

- budynki_polaczone_all.gpkg – warstwę przestrzenną przedstawiającą obrysy budynków.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ps = pd.read_csv("EGMS_Bytom_results.csv")
buildings = gpd.read_file("budynki_polaczone_all.gpkg")

print("Liczba PS:", len(ps))
print("Liczba budynków:", len(buildings))
print("CRS budynków:", buildings.crs)

## Usunięcie powtarzających się geometrii

In [ ]:
# Usunięcie identycznych geometrii
buildings = (
    buildings
    .drop_duplicates(subset="geometry")
    .reset_index(drop=True)
    .copy()
)

# Nadanie identyfikatorów budynkom
buildings["building_id"] = np.arange(
    1, len(buildings) + 1
)

print("Liczba budynków:", len(buildings))

## Ujednolicenie układów współrzędnych

In [ ]:
ps_gdf = gpd.GeoDataFrame(
    ps.copy(),
    geometry=gpd.points_from_xy(
        ps["easting"],
        ps["northing"]
    ),
    crs="EPSG:3035"
)

# Transformacja budynków do układu punktów EGMS
buildings = buildings.to_crs(ps_gdf.crs)

print("CRS PS:", ps_gdf.crs)
print("CRS budynków:", buildings.crs)

## Wybór geometrii i atrybutów

In [ ]:
building_footprints = buildings[
    ["building_id", "geometry"]
].copy()

print("Typy geometrii:")
print(building_footprints.geometry.geom_type.value_counts())

In [ ]:
ps_for_join = ps_gdf[
    [
        "pid",
        "mean_velocity",
        "acceleration",
        "trend_change",
        "anomaly_score",
        "anomaly",
        "cluster",
        "rmse_ts",
        "geometry"
    ]
].copy()

## Przypisanie najbliższego PS


In [ ]:
MAX_DISTANCE = X

buildings_ps_all = gpd.sjoin_nearest(
    building_footprints,
    ps_for_join,
    how="left",
    max_distance=MAX_DISTANCE,
    distance_col="distance_PS"
)

In [ ]:
buildings_ps = (
    buildings_ps_all
    .reset_index(drop=True)
    .sort_values(
        ["building_id", "distance_PS", "anomaly_score"],
        ascending=[True, True, False],
        na_position="last"
    )
    .drop_duplicates(
        subset="building_id",
        keep="first"
    )
    .reset_index(drop=True)
)

buildings_ps["has_PS_30m"] = (
    buildings_ps["pid"].notna()
)

In [ ]:
matched = buildings_ps["has_PS_30m"]

print("Wszystkie budynki:", len(buildings_ps))
print("Budynki z PS do 30 m:", matched.sum())
print("Budynki bez PS do 30 m:", (~matched).sum())
print("Wykorzystane unikalne PS:", buildings_ps["pid"].nunique())
print(
    "PS wewnątrz budynku:",
    (buildings_ps["distance_PS"] == 0).sum()
)
print(
    "Mediana odległości:",
    round(
        buildings_ps.loc[
            matched,
            "distance_PS"
        ].median(),
        1
    ),
    "m"
)

## Utworzenie warstwy wynikowej

In [ ]:
result = buildings.merge(
    buildings_ps.drop(
        columns=["geometry", "index_right"],
        errors="ignore"
    ),
    on="building_id",
    how="left"
)

result = gpd.GeoDataFrame(
    result,
    geometry="geometry",
    crs=buildings.crs
)

print("Budynki w wyniku:", len(result))
print("Budynki z przypisanym PS:", result["has_PS_30m"].sum())
print("Unikalne identyfikatory:", result["building_id"].nunique())

In [ ]:
result.to_file(
    "budynki_EGMS_results.gpkg",
    layer="budynki_EGMS",
    driver="GPKG"
)

print("Zapisano: budynki_EGMS_results.gpkg")